# Random forest
## Baseline

In [37]:
from pathlib import Path
import json
import pandas as pd
import os

DATA_DIR = Path("lasftm_asia")
FEATURES_JSON = DATA_DIR / "lastfm_asia_features.json"
TARGET_CSV = DATA_DIR / "lastfm_asia_target.csv"
OUT_ONEHOT_CSV = DATA_DIR / "lastfm_asia_features_onehot.csv"


if os.path.exists(OUT_ONEHOT_CSV):
    onehot_df = pd.read_csv(OUT_ONEHOT_CSV)

else:


    with FEATURES_JSON.open("r") as f:
        raw_features = json.load(f)

    rows = [
        (int(user_id), int(artist_id), 1)
        for user_id, artist_ids in raw_features.items()
        for artist_id in artist_ids
    ]
    long_df = pd.DataFrame(rows, columns=["user_id", "artist_id", "value"])

    # Pivot to one-hot format:
    onehot_df = (
        long_df.pivot_table(
            index="user_id",
            columns="artist_id",
            values="value",
            aggfunc="max",
            fill_value=0,
        )
        .astype("int8")
        .sort_index()
    )

    # Rename columns
    onehot_df.columns = [f"artist_{int(c)}" for c in onehot_df.columns]
    onehot_df = onehot_df.reset_index()

    target_df = pd.read_csv(TARGET_CSV)
    onehot_df = onehot_df.merge(target_df, how="inner", left_on="user_id", right_on="id").drop(columns=["id"])

    onehot_df.to_csv(OUT_ONEHOT_CSV, index=False)

print(f"Saved: {OUT_ONEHOT_CSV}")
print(f"Shape: {onehot_df.shape}")
onehot_df.head()


Saved: lasftm_asia/lastfm_asia_features_onehot.csv
Shape: (7451, 7844)


,user_id,artist_0,artist_1,artist_2,artist_3,artist_4,artist_5,artist_6,artist_7,artist_8,...,artist_7833,artist_7834,artist_7835,artist_7836,artist_7837,artist_7838,artist_7839,artist_7840,artist_7841,target
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,8
1,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,17
2,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
3,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,17
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

X = onehot_df.drop(columns=["target"]).to_numpy()
y = onehot_df["target"].to_numpy()

rf_model = RandomForestClassifier(n_jobs=-1)



rf_scores = cross_validate(rf_model, X, y, cv=15, scoring=["f1_macro", "accuracy"])



In [39]:
print(f"Random Forest:")
print(f"    F1 macro: {rf_scores["test_f1_macro"].mean():.4f} +- {rf_scores["test_f1_macro"].std():.4f}")
print(f"    Accuracy: {rf_scores["test_accuracy"].mean():.4f} +- {rf_scores["test_accuracy"].std():.4f}")


Random Forest:
    F1 macro: 0.4162 +- 0.0220
    Accuracy: 0.7345 +- 0.0141


# Weighting

In [40]:
import numpy as np
from scipy.sparse import csr_matrix

SVD_DIM = 2048
SEED = 42

with FEATURES_JSON.open("r") as f:
    raw_features = json.load(f)

target_df = pd.read_csv(TARGET_CSV)


rows, cols = [], []

for user_id, artist_ids in raw_features.items():
    u = int(user_id)
    for a in artist_ids:
        rows.append(u)
        cols.append(int(a))

data = np.ones(len(rows), dtype=np.float32)

N = max(rows) + 1
D = max(cols) + 1

X_bin = csr_matrix((data, (rows, cols)), shape=(N, D))

In [41]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

# (a) frequency-weighted + L2 normalise (no SVD)
df_vec   = np.asarray(X_bin.sum(axis=0)).flatten()
X_freq   = normalize(X_bin.multiply(df_vec), norm="l2")   # sparse (N, D)



In [42]:
target_df = target_df.sort_values("id")

X = X_freq[target_df["id"].values]
y = target_df["target"].values


In [43]:

rf_model = RandomForestClassifier(n_jobs=-1)

lr_model = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight="balanced"
)

rf_scores = cross_validate(
    rf_model,
    X,
    y,
    cv=5,
    scoring=["f1_macro", "accuracy"],
    n_jobs=-1
)
lr_scores = cross_validate(
    lr_model,
    X,
    y,
    cv=5,
    scoring=["f1_macro", "accuracy"],
    n_jobs=-1
)

print(f"Random Forest:")
print(f"    F1 macro: {rf_scores["test_f1_macro"].mean():.4f} +- {rf_scores["test_f1_macro"].std():.4f}")
print(f"    Accuracy: {rf_scores["test_accuracy"].mean():.4f} +- {rf_scores["test_accuracy"].std():.4f}")

print(f"Logistic Regression:")
print(f"    F1 macro: {lr_scores["test_f1_macro"].mean():.4f} +- {lr_scores["test_f1_macro"].std():.4f}")
print(f"    Accuracy: {lr_scores["test_accuracy"].mean():.4f} +- {lr_scores["test_accuracy"].std():.4f}")



/opt/pyenvs/ina/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/pyenvs/ina/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/pyenvs/ina/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/pyenvs/ina/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecifie

Random Forest:
    F1 macro: 0.3956 +- 0.0030
    Accuracy: 0.7104 +- 0.0061
Logistic Regression:
    F1 macro: 0.3773 +- 0.0091
    Accuracy: 0.5392 +- 0.0131


# SVD

In [44]:


# -----------------------
# SVD embedding (user latent vectors)
# -----------------------
svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
X_embed = svd.fit_transform(X_tfidf).astype(np.float32)

X_embed = normalize(X_embed, norm="l2")


print("Explained variance:", svd.explained_variance_ratio_.sum())

KeyboardInterrupt: 

In [ ]:
target_df = target_df.sort_values("id")

X = X_embed[target_df["id"].values]
y = target_df["target"].values


In [ ]:
rf_model = RandomForestClassifier(n_jobs=-1)

lr_model = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight="balanced"
)

rf_scores = cross_validate(
    rf_model,
    X,
    y,
    cv=5,
    scoring=["f1_macro", "accuracy"],
    n_jobs=-1
)
lr_scores = cross_validate(
    rf_model,
    X,
    y,
    cv=5,
    scoring=["f1_macro", "accuracy"],
    n_jobs=-1
)



In [ ]:

print(f"Random Forest:")
print(f"    F1 macro: {rf_scores["test_f1_macro"].mean():.4f} +- {rf_scores["test_f1_macro"].std():.4f}")
print(f"    Accuracy: {rf_scores["test_accuracy"].mean():.4f} +- {rf_scores["test_accuracy"].std():.4f}")

print(f"Logistic Regression:")
print(f"    F1 macro: {lr_scores["test_f1_macro"].mean():.4f} +- {lr_scores["test_f1_macro"].std():.4f}")
print(f"    Accuracy: {lr_scores["test_accuracy"].mean():.4f} +- {lr_scores["test_accuracy"].std():.4f}")



Random Forest:
    F1 macro: 0.2184 +- 0.0080
    Accuracy: 0.5226 +- 0.0091
Logistic Regression:
    F1 macro: 0.2171 +- 0.0093
    Accuracy: 0.5258 +- 0.0109
